
# Applied Machine Learning, Notebook 7
## Probability calibration: when the model says 30%, it has to be 30%

**Course:** *Applied Machine Learning* · **Prepared by:** David Díaz, with assistance from Claude (Anthropic) · **Environment:** Google Colab

### Where we come from

In notebooks 5 and 6 we said the same sentence several times: "the model stops being calibrated". We said it when
balancing the classes (the mean probability in testing rose from 0.33 to 0.48 when the actual default rate
was 0.30), we said it when comparing the logistic with the boosting in Taiwan, and we said it when explaining why the
optimal threshold from the cost formula did not match the one from the table. We never showed how it gets fixed. This
notebook closes that pending item.

The idea is simple. A model that predicts probabilities does two different things: it **orders** the customers
(the riskiest at the top) and it puts a **number** on them. The AUC measures only the first. Calibration measures the
second: if I take all the customers the model gave 30% to, do 30% of them default? To
approve or reject with an arbitrary threshold, ordering well is enough; to set a threshold with costs, to
provision, to charge a rate according to risk, or to add up the risks of a portfolio, the number has to
mean what it says.

### What you will learn today

1. How calibration is measured: the reliability diagram, the Brier score and its decomposition, the log loss and the expected calibration error (ECE).
2. Why it is lost: when balancing classes or weighting rows, because of the model's form (a misspecified logistic), and in trees and ensembles by construction.
3. How it is corrected when the cause is the balancing: the analytical intercept correction, exact for the logistic and with no extra data.
4. How it is corrected in general: Platt scaling and isotonic regression, always on data the model did not see, and what `CalibratedClassifierCV` does.
5. What changes in the decision: with calibrated probabilities, the threshold from the cost formula becomes reliable again, and the probability is useful for provisioning and for pricing risk.


In [ ]:

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings, time

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss

warnings.filterwarnings("ignore")
pd.set_option("display.width", 130)



## 1. The two datasets, prepared the same way as before

We use the same two datasets from notebooks 5 and 6, with the same preparation and the same split, so that the
numbers can be compared cell by cell. **German Credit**: 1,000 loans, 300 defaults, 18 columns, 700
in training and 300 in testing. **Taiwan**: 30,000 cards, 22% default, 23 columns, 21,000 in training and
9,000 in testing. The loading cells are the usual ones; if you have the course CSVs, paste their links.


In [ ]:

URL_GERMAN = ""    # <-- paste here the link to the course CSV (german_credit.csv) if you have it

def normalizar_link(url):
    '''Converts a Dropbox or Google Drive link into a direct-download link.'''
    if "dropbox.com" in url:
        return url.replace("dl=0", "dl=1").replace("www.dropbox.com", "dl.dropboxusercontent.com")
    if "drive.google.com" in url and "/d/" in url:
        return "https://drive.google.com/uc?export=download&id=" + url.split("/d/")[1].split("/")[0]
    return url

if URL_GERMAN:
    german = pd.read_csv(normalizar_link(URL_GERMAN))
else:
    german = fetch_openml("credit-g", version=1, as_frame=True).frame
    for columna in german.select_dtypes("category").columns:
        german[columna] = german[columna].astype(str)
german = german.reset_index(drop=True)
german.insert(0, "id", np.arange(1, len(german) + 1))
print("Customers:", len(german), "  Columns:", german.shape[1] - 1)
print("Proportion of bad payers:", round((german["class"] == "bad").mean(), 3))
german[["id", "duration", "credit_amount", "age", "checking_status", "credit_history", "savings_status", "class"]].head(8)


In [ ]:

T = pd.DataFrame({"id": german["id"], "y": (german["class"] == "bad").astype(int)})
T["duracion"] = german["duration"].astype(float)                     # months
T["monto"] = german["credit_amount"].astype(float)                   # German marks (DM)
T["edad"] = german["age"].astype(float)
T["cuota_pct_ingreso"] = german["installment_commitment"].astype(float)   # bracket 1 to 4 of the installment as % of income

cs = german["checking_status"]                                       # base: no checking account
T["cta_negativa"] = (cs == "<0").astype(int)
T["cta_0_200"] = (cs == "0<=X<200").astype(int)
T["cta_sobre_200"] = (cs == ">=200").astype(int)
ch = german["credit_history"]                                        # base: existing credits paid on time
T["hist_critico"] = (ch == "critical/other existing credit").astype(int)
T["hist_atrasos"] = (ch == "delayed previously").astype(int)
T["hist_sin_creditos"] = ch.isin(["all paid", "no credits/all paid"]).astype(int)
sv = german["savings_status"]                                        # base: savings below 100 DM
T["ahorro_100_1000"] = sv.isin(["100<=X<500", "500<=X<1000"]).astype(int)
T["ahorro_sobre_1000"] = (sv == ">=1000").astype(int)
T["ahorro_desconocido"] = (sv == "no known savings").astype(int)
hs = german["housing"]                                               # base: owns home
T["vivienda_arriendo"] = (hs == "rent").astype(int)
T["vivienda_gratis"] = (hs == "for free").astype(int)
pu = german["purpose"]                                               # base: other purposes
T["proposito_auto"] = pu.isin(["new car", "used car"]).astype(int)
T["proposito_radio_tv"] = (pu == "radio/tv").astype(int)
T["proposito_muebles"] = (pu == "furniture/equipment").astype(int)

XCOLS = [c for c in T.columns if c not in ("id", "y")]
NUM = ["duracion", "monto", "edad", "cuota_pct_ingreso"]
print(len(XCOLS), "explanatory columns:", XCOLS)
T.head()


In [ ]:

rng = np.random.default_rng(2026)
conjunto = np.array(["training"] * len(T), dtype=object)
for clase in (0, 1):
    idx = np.where(T["y"].values == clase)[0]
    rng.shuffle(idx)
    conjunto[idx[: int(round(0.3 * len(idx)))]] = "testing"
T["conjunto"] = conjunto
tr = T["conjunto"] == "training"
te = ~tr

media = T.loc[tr, NUM].mean()
desv = T.loc[tr, NUM].std(ddof=0)      # population standard deviation, the same one StandardScaler uses
S = T.copy()
S[NUM] = (T[NUM] - media) / desv

X_tr, y_tr = S.loc[tr, XCOLS].values, S.loc[tr, "y"].values
X_te, y_te = S.loc[te, XCOLS].values, S.loc[te, "y"].values
print("training:", tr.sum(), "customers,", y_tr.sum(), "bad payers  |  testing:", te.sum(), "customers,", y_te.sum(), "bad payers")
pd.DataFrame({"training mean": media.round(3), "training std": desv.round(3)})


In [ ]:

URL_TAIWAN = ""    # <-- paste here the link to the course CSV (taiwan_credit.csv) if you have it

def normalizar_link(url):
    if "dropbox.com" in url:
        return url.replace("dl=0", "dl=1").replace("www.dropbox.com", "dl.dropboxusercontent.com")
    if "drive.google.com" in url and "/d/" in url:
        return "https://drive.google.com/uc?export=download&id=" + url.split("/d/")[1].split("/")[0]
    return url

if URL_TAIWAN:
    tw = pd.read_csv(normalizar_link(URL_TAIWAN))
    X_tw = tw.drop(columns=tw.columns[-1]).apply(pd.to_numeric)
    y_tw = tw[tw.columns[-1]].astype(int).values
else:
    tw = fetch_openml("default-of-credit-card-clients", version=1, as_frame=True)
    X_tw = tw.data.apply(pd.to_numeric)
    y_tw = tw.target.astype(int).values

meses = ["sep", "ago", "jul", "jun", "may", "abr"]
X_tw.columns = (["limite_credito", "sexo", "educacion", "estado_civil", "edad"] + ["atraso_" + m for m in meses]
             + ["factura_" + m for m in meses] + ["pago_" + m for m in meses])
print("Customers:", len(X_tw), "  Columns:", X_tw.shape[1], "  Proportion that defaulted:", round(y_tw.mean(), 3))



In [ ]:

X_tw_tr, X_tw_te, yt_tr, yt_te = train_test_split(X_tw, y_tw, test_size=0.3, random_state=42, stratify=y_tw)
esc_tw = StandardScaler().fit(X_tw_tr)
A_tr, A_te = esc_tw.transform(X_tw_tr), esc_tw.transform(X_tw_te)
print("Taiwan, training:", len(yt_tr), "customers,", int(yt_tr.sum()), "defaulted  |  testing:", len(yt_te), "customers,", int(yt_te.sum()), "defaulted")



## 2. What being calibrated means and how it is measured

A model is **calibrated** if, among the customers it assigns a probability $p$ to, the fraction that
actually turns out positive is $p$. With continuous probabilities there is no "everyone who has exactly
0.30", so they are grouped: the testing customers are sorted by $p$, split into ten groups of the same
size (deciles), and in each group the mean $p$ is compared with the actual fraction of positives. Plotting those
ten points against the diagonal is the **reliability diagram**: points on the diagonal, calibrated; below
it, the model exaggerates the risk; above it, it underestimates it.

Three numbers summarize the diagram:

- **Brier score** $= \frac{1}{n}\sum_i (p_i - y_i)^2$, the mean squared error of the probability against the
  0/1 outcome. It decomposes into a calibration term (how far from the diagonal the points are)
  plus a resolution term (how different the deciles are from each other) plus the variance of $y$, which does not
  depend on the model. Lowering the Brier can come from calibrating better or from discriminating better.
- **Log loss**, the mean cross-entropy from notebook 5: it punishes misplaced confidence hard. A
  model that says 0.0001 and misses pays 9.2; one that says 0.01 pays 4.6.
- **ECE** (expected calibration error): the average, weighted by the size of each decile, of the vertical
  distance between the point and the diagonal. It is the most direct way to put a number on the diagram. With ten
  deciles of 30 customers (German Credit) it is quite noisy; with deciles of 900 (Taiwan) it is reliable.

The rule that none of the three breaks: the AUC does not change if I apply any increasing transformation
to the probabilities (the order is the same), but the Brier, the log loss and the ECE do. That is why two
models can have the same AUC and very different calibrations, and why calibrating never improves the order.


In [ ]:

def confiabilidad(y_real, p, k=10):
    '''Table by deciles of p: how many customers, mean p, actual fraction of positives.'''
    q = pd.qcut(p, k, labels=False, duplicates="drop")
    return pd.DataFrame({"decil": q, "p": p, "y": y_real}).groupby("decil").agg(n=("y", "size"), p_promedio=("p", "mean"), fraccion_real=("y", "mean"))

def ece(y_real, p, k=10):
    t = confiabilidad(y_real, p, k)
    return float((t.n / len(y_real) * (t.fraccion_real - t.p_promedio).abs()).sum())

def metricas(y_real, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return dict(AUC=roc_auc_score(y_real, p), Brier=brier_score_loss(y_real, p), log_loss=log_loss(y_real, p), ECE=ece(y_real, p), p_promedio=p.mean(), tasa_real=y_real.mean())

def diagrama(y_real, modelos, titulo):
    '''Reliability diagram: one trace per model, {name: p}.'''
    fig = go.Figure()
    fig.add_scatter(x=[0, 1], y=[0, 1], mode="lines", name="perfect calibration", line=dict(dash="dash", color="gray"))
    for nombre, p in modelos.items():
        t = confiabilidad(y_real, p)
        fig.add_scatter(x=t.p_promedio, y=t.fraccion_real, mode="lines+markers", name=nombre)
    fig.update_layout(title=titulo, xaxis_title="predicted probability (decile mean)", yaxis_title="actual fraction of positives",
                      xaxis_range=[0, 1], yaxis_range=[0, 1], height=460, width=620)
    return fig

def tabla_metricas(y_real, modelos):
    return pd.DataFrame({n: metricas(y_real, p) for n, p in modelos.items()}).T.round(4)

base = LogisticRegression(penalty=None, max_iter=5000).fit(X_tr, y_tr)
p_base = base.predict_proba(X_te)[:, 1]
print(confiabilidad(y_te, p_base).round(3))
print()
print(pd.Series(metricas(y_te, p_base)).round(4))
diagrama(y_te, {"unbalanced logistic": p_base}, "German Credit, testing: reliability diagram").show()



The logistic from notebook 5 is reasonably calibrated: the mean probability in testing (0.332) is
close to the actual rate (0.300), the low and high deciles fall near the diagonal, and the ECE of 0.052 is
compatible with the noise of deciles of 30 customers. It is what you expect from a logistic trained with
unweighted cross-entropy: the cost function it minimizes is exactly the one that rewards calibration.
That is going to change as soon as we weight it.



## 3. Why calibration is lost

### Cause 1: balancing the classes

In notebook 5 we corrected the class imbalance with `class_weight="balanced"` (weight 1.667 for the bad payers
and 0.714 for the good ones) and we saw that recall went up. The price, which we mentioned but did not show, is that the model
learns from a fictitious dataset where bad payers are 50%, and returns probabilities from that fictitious
dataset. The order of the customers barely changes (the AUC is the same), but every number is inflated.


In [ ]:

cw = LogisticRegression(penalty=None, max_iter=5000, class_weight="balanced").fit(X_tr, y_tr)
p_cw = cw.predict_proba(X_te)[:, 1]
n1, n0 = int(y_tr.sum()), int((y_tr == 0).sum())
w1, w0 = len(y_tr) / (2 * n1), len(y_tr) / (2 * n0)
print(f"'balanced' weights: w1 = {w1:.3f}, w0 = {w0:.3f}, ratio w1/w0 = {w1/w0:.3f} (= n0/n1)")
print(tabla_metricas(y_te, {"unbalanced": p_base, "class weights": p_cw}))
diagrama(y_te, {"unbalanced": p_base, "class weights": p_cw}, "German Credit: the balanced model inflates every probability").show()



The diagram says it all: the class weights curve lies entirely below the diagonal (the model claims
more risk than there is), the ECE jumps from 0.05 to 0.18 and the Brier from 0.151 to 0.188, with the same AUC. The
same happens with oversampling (repeating rows is weighting them) and with undersampling. It is not a defect of
balancing: it is its logical consequence, because we changed the model's base rate.

### Cause 2: the form of the model

A linear logistic assumes that the log odds are linear in the variables. If reality has
steps, as in Taiwan (default jumps from 13% to 33% with one month of delay and to 68% with two), the logistic
draws a ramp that passes above reality in one stretch and below it in another. It is misspecified,
and a misspecified model can be miscalibrated even though it was trained with cross-entropy.

### Cause 3: trees and ensembles, by construction

An unlimited tree ends up with one record per leaf and predicts 0 or 1: extreme probabilities that are almost
always wrong, and the log loss punishes them without mercy. A random forest averages hundreds of trees that
memorize, and that average tends to compress the probabilities toward the center or to exaggerate them at the
extremes, depending on how the trees grew. Boosting, which minimizes cross-entropy in each round,
usually comes out better calibrated, but has no guarantee either.

To see all three things at once we train on Taiwan the logistic, an unlimited tree, the depth-4
tree, two random forests (one with leaves of 1 customer and another with leaves of 5, the one from notebook 6) and the
boosting. A detail that matters from now on: for section 5 we are going to need data the model has not
seen and that is not testing, so training is split in two from here: **fit** (70% of training) and
**calibration** (30%). Every model in this notebook is trained on the fit part.


In [ ]:

A_fit, A_cal, y_fit, y_cal = train_test_split(A_tr, yt_tr, test_size=0.3, random_state=1, stratify=yt_tr)
print("fit:", len(y_fit), " calibration:", len(y_cal), " testing:", len(yt_te))

modelos_tw = {
    "logistic": LogisticRegression(penalty=None, max_iter=5000),
    "unlimited tree": DecisionTreeClassifier(random_state=0),
    "tree depth 4": DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, random_state=0),
    "random forest, leaves of 1": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=0),
    "random forest, leaves of 5": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=0),
    "gradient boosting": GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=0),
}
P_cal, P_te = {}, {}
t0 = time.time()
for nombre, m in modelos_tw.items():
    m.fit(A_fit, y_fit)
    P_cal[nombre] = m.predict_proba(A_cal)[:, 1]
    P_te[nombre] = m.predict_proba(A_te)[:, 1]
print(f"({time.time() - t0:.0f} s)")
print(tabla_metricas(yt_te, P_te))
diagrama(yt_te, {k: P_te[k] for k in ["logistic", "random forest, leaves of 1", "gradient boosting"]}, "Taiwan, testing: three uncalibrated models").show()
print(confiabilidad(yt_te, P_te["logistic"]).round(3))



Three readings. **The logistic is the worst calibrated** of those that predict something (ECE 0.051): in the middle
deciles it says 18% to 23% and reality is 12% to 15%, and in the top decile it says 59% and 66% happens; it is the ramp
against the staircase. **The unlimited tree** is a calibration disaster even though its ECE by deciles looks
low (it predicts 0 or 1, so the deciles cannot be built; look at the Brier of 0.27 and the log loss of 3.7,
against 0.14 and 0.45 for the others). **The ensembles are better than one would expect**: the boosting and the
random forest with leaves of 5 have ECE of 0.012 to 0.014, and the random forest with leaves of 1 exaggerates a little
at the ends (ECE 0.030). With 14,700 fit rows and reasonable minimum leaves, the ensembles in this
problem come out fairly well calibrated on their own. It is not always like that, and that is why you have to look at the diagram before
deciding whether correcting is worth it.



## 4. Analytical intercept correction: undoing the balancing with no extra data

When the miscalibration comes from having weighted the classes, the logistic has an exact solution that does not
need a single extra data point. Weighting class 1 by $w_1$ and class 0 by $w_0$ is equivalent to training on a
population where the prior odds of being positive were multiplied by $w_1 / w_0$. In a logistic that
shifts only the intercept: the logit the balanced model learns is the true logit plus
$\ln(w_1 / w_0)$. To go back to the real population it is enough to subtract that constant:

$$z_{\text{corregido}} = z_{\text{balanceado}} - \ln\frac{w_1}{w_0}, \qquad p_{\text{corregido}} = \frac{1}{1 + e^{-z_{\text{corregido}}}}$$

With `class_weight="balanced"` the ratio $w_1 / w_0 = n_0 / n_1 = 490/210 = 2.333$ and the constant is
$\ln 2.333 = 0.847$. With oversampling of the minority class repeated $k$ times the constant is $\ln k$; with
undersampling that leaves $n_0'$ good payers, $\ln(n_0 / n_0')$. The order of the customers does not change (it is subtracting a
constant from the logit), so the AUC stays the same and only the level is corrected. It is the result of King and Zeng
(2001) for choice-based sampling, and in practice it is the first thing to do if you balanced a logistic.


In [ ]:

z_cw = cw.decision_function(X_te)                      # the logit of the balanced model
p_corr = 1 / (1 + np.exp(-(z_cw - np.log(w1 / w0))))   # subtract ln(w1/w0) = ln(490/210)
print(f"constant subtracted: ln({w1/w0:.3f}) = {np.log(w1/w0):.3f}")
print(tabla_metricas(y_te, {"unbalanced": p_base, "class weights": p_cw, "class weights + correction": p_corr}))
diagrama(y_te, {"unbalanced": p_base, "class weights": p_cw, "class weights + correction": p_corr}, "German Credit: the intercept correction brings the curve back to the diagonal").show()

# the same with oversampling: each bad payer twice is equivalent to w1/w0 = 2, constant ln 2
over = LogisticRegression(penalty=None, max_iter=5000).fit(X_tr, y_tr, sample_weight=np.where(y_tr == 1, 2.0, 1.0))
z_over = over.decision_function(X_te)
print()
print(tabla_metricas(y_te, {"oversampling x2": 1 / (1 + np.exp(-z_over)), "oversampling x2 + correction ln 2": 1 / (1 + np.exp(-(z_over - np.log(2))))}))



The corrected curve falls on top of the unbalanced model's: same AUC of 0.824, Brier back to 0.151 and
ECE 0.045, with mean probability 0.332. We balanced to move the operating point and in the end we
recover the real probabilities: you can have both. (And it is another way of seeing the conclusion
of notebook 5: for a logistic, balancing and then correcting the intercept is the same as not balancing and
moving the threshold.)



## 5. General correction: Platt scaling and isotonic regression

When the cause is not a known weight (the misspecified logistic, the trees, a model that comes from
somewhere else), there is no constant to subtract. The general solution is to **learn a function that translates the
model's scores into real probabilities**, using labeled data the model did not see. Two functions are
the usual ones:

- **Platt scaling** (Platt, 1999): a logistic with a single input, the model's logit, $p = 1/(1 + e^{-(a z + b)})$.
  Two parameters, so it works with little data, and it can only stretch or compress and shift: it fixes a
  model that is uniformly too confident or too lukewarm, but it cannot fix a ramp against a
  staircase, because the transformation is a sigmoid.
- **Isotonic regression** (Zadrozny and Elkan, 2002): the increasing step function that best fits the
  actual fraction of positives against the score. It has no fixed shape, so it can correct any
  monotone deformation, but it needs thousands of rows and can overfit with few.

The non-negotiable rule: the calibration function is fitted on the **calibration** part (or by
cross-validation), never on the rows the model was fitted with (on those the model looks better than it
is) and never on testing (which has to stay clean for measuring). That is why we split training in two in
section 3. In scikit-learn, `CalibratedClassifierCV(modelo, method="sigmoid" | "isotonic", cv=5)` does
this by cross-validation and also averages the five models, so its results mix calibration
with a bit of ensembling; here we do it by hand so that each piece can be seen.


In [ ]:

def logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))

def calibrar_platt(p_cal, y_cal, p_nuevo):
    f = LogisticRegression().fit(logit(p_cal).reshape(-1, 1), y_cal)      # a·z + b
    return f.predict_proba(logit(p_nuevo).reshape(-1, 1))[:, 1]

def calibrar_isotonica(p_cal, y_cal, p_nuevo):
    f = IsotonicRegression(out_of_bounds="clip").fit(p_cal, y_cal)
    return f.predict(p_nuevo)

filas = []
P_platt, P_iso = {}, {}
for nombre in modelos_tw:
    P_platt[nombre] = calibrar_platt(P_cal[nombre], y_cal, P_te[nombre])
    P_iso[nombre] = calibrar_isotonica(P_cal[nombre], y_cal, P_te[nombre])
    for etiqueta, p in [("uncalibrated", P_te[nombre]), ("Platt", P_platt[nombre]), ("isotonic", P_iso[nombre])]:
        m = metricas(yt_te, p); filas.append(dict(modelo=nombre, calibración=etiqueta, AUC=m["AUC"], Brier=m["Brier"], log_loss=m["log_loss"], ECE=m["ECE"]))
tabla_cal = pd.DataFrame(filas).set_index(["modelo", "calibración"]).round(4)
print(tabla_cal.to_string())


In [ ]:

diagrama(yt_te, {"logistic": P_te["logistic"], "logistic + Platt": P_platt["logistic"], "logistic + isotonic": P_iso["logistic"]},
         "Taiwan: the misspecified logistic, Platt does not fix it and isotonic does").show()
diagrama(yt_te, {"random forest, leaves of 1": P_te["random forest, leaves of 1"], "+ Platt": P_platt["random forest, leaves of 1"], "+ isotonic": P_iso["random forest, leaves of 1"]},
         "Taiwan: the random forest with leaves of 1 exaggerates at the ends, and both corrections bring it closer").show()



How to read the table and the two diagrams:

- **The logistic**: Platt does nothing (ECE 0.053 against 0.051), because a sigmoid on top of a sigmoid is still
  a ramp. Isotonic does: ECE from 0.051 to 0.009, Brier from 0.146 to 0.143, and the diagram sticks to the
  diagonal. It is the shape correction that Platt cannot make. Notice that the AUC barely changes (0.715 to
  0.717): calibrating does not improve the order, and the small difference comes from the ties the staircase creates.
- **The unlimited tree**: Brier from 0.27 to 0.16 and log loss from 3.7 to 0.50 with either of the two, but the
  AUC stays at 0.616. Calibration fixes the numbers, not the ranking; a model that orders badly keeps
  ordering badly with pretty probabilities.
- **Random forest with leaves of 1**: ECE from 0.030 to 0.019 (Platt) and 0.013 (isotonic), with the Brier barely better.
- **The ones that were already calibrated** (depth-4 tree, random forest with leaves of 5, boosting): the
  corrections change the Brier in the fourth digit and isotonic sometimes worsens the log loss. Calibrating a
  model that does not need it is not free: the calibration function also has sampling error.

The practical rule that comes out of here: first the diagram; if the curve is far from the diagonal as a
uniform shift, Platt (or the intercept correction if the cause was a weight); if it is deformed and
there are thousands of rows, isotonic; if it is stuck to the diagonal, do not touch it.



## 6. What changes in the decision: the cost formula works again

In notebook 5 we saw that with a false negative five times more expensive than a false positive the optimal threshold
from the **formula** is $c_{FP} / (c_{FP} + c_{FN}) = 1/6 = 0.167$, and that the **table** on testing gave another
number. We said that one of the reasons was calibration: the formula assumes that $p$ is the real probability
of default, and if $p$ is inflated, the formula's threshold cuts where it should not. Now we can check it
with the balanced model, before and after the correction, and with the Taiwan random forest, before and after
Platt.


In [ ]:

def costo_por_umbral(y_real, p, c_fn, c_fp, umbrales=np.round(np.arange(0.05, 0.96, 0.05), 2)):
    return pd.Series({u: int(((p > u) & (y_real == 0)).sum() * c_fp + ((p <= u) & (y_real == 1)).sum() * c_fn) for u in umbrales})

def comparar_umbrales(y_real, modelos, c_fn, c_fp):
    u_formula = c_fp / (c_fp + c_fn)
    filas = []
    for nombre, p in modelos.items():
        tabla = costo_por_umbral(y_real, p, c_fn, c_fp)
        costo_formula = int(((p > u_formula) & (y_real == 0)).sum() * c_fp + ((p <= u_formula) & (y_real == 1)).sum() * c_fn)
        filas.append(dict(modelo=nombre, umbral_formula=round(u_formula, 3), costo_en_formula=costo_formula, umbral_tabla=float(tabla.idxmin()), costo_minimo_tabla=int(tabla.min()),
                          p_promedio=round(p.mean(), 3)))
    return pd.DataFrame(filas).set_index("modelo")

print("German Credit, c_FN = 5, c_FP = 1:")
print(comparar_umbrales(y_te, {"unbalanced": p_base, "class weights": p_cw, "class weights + correction": p_corr}, c_fn=5, c_fp=1))
print()
print("Taiwan, c_FN = 4, c_FP = 1:")
print(comparar_umbrales(yt_te, {"random forest, leaves of 1": P_te["random forest, leaves of 1"], "+ Platt": P_platt["random forest, leaves of 1"], "+ isotonic": P_iso["random forest, leaves of 1"],
                                "gradient boosting": P_te["gradient boosting"], "gradient boosting + Platt": P_platt["gradient boosting"]}, c_fn=4, c_fp=1))



With the balanced model without correction, the formula (0.167) is a disaster: it cuts very low on an
inflated scale and the cost rises to 182, while the table finds its minimum only at 0.50. With the intercept corrected,
the formula again gives a cost (149) close to the table's minimum (133 at 0.25), just as with the
unbalanced model. The difference that remains between 0.167 and 0.25 is the other reason we gave in notebook 5: the
sampling variation of a table built with 300 customers.

In Taiwan, with 9,000 testing customers, the table is much less noisy and the models were already almost
calibrated, so the story is different and also instructive. For the boosting, formula and table say the
same (0.20) before and after Platt: there was nothing to correct. For the random forest with leaves of 1 the
formula says 0.20 and the table 0.25, and calibrating does not move the table's minimum, because the miscalibration was
small (ECE 0.03) and was at the ends, not around 0.2. But look at the costs: between 0.20 and 0.25 the
difference is 4,732 against 4,650 uncalibrated and 4,722 against 4,658 with Platt, less than 2%. The cost
curve is flat near the optimum, and with a calibrated model the formula falls inside that flat zone even if it does not
give the exact minimum of the table. That is the concrete usefulness of calibrating: with probabilities that mean what
they say, the threshold can be set with pencil and paper from the costs, without depending on the testing
sample, and the same probability is useful for provisioning and for pricing risk. And when the
miscalibration is large, as in the balanced German Credit model, the uncalibrated formula does not fall in
any flat zone: it falls far away.



## 7. What to take away

1. A classifier does two things: order and put a number on the risk. The AUC measures the first; the reliability
   diagram, the Brier, the log loss and the ECE measure the second. Calibrating never improves the order.
2. Calibration is lost through three paths: weighting or resampling classes, a wrong model form,
   and trees and ensembles by construction. A logistic trained without weights and well specified is born
   calibrated.
3. If you balanced a logistic, subtract $\ln(w_1/w_0)$ from the logit: exact, free and with no extra data.
4. In the general case, Platt (two parameters, corrects level and scale) or isotonic (corrects shape, needs
   thousands of rows), always fitted on data the model did not see and that is not testing.
5. Look at the diagram before calibrating: a model that is already calibrated does not improve and can get worse.
6. With calibrated probabilities, the cost threshold can be set with the formula, and the probability is useful
   for provisioning and for pricing risk, not only for ordering.



## Exercises

1. **Undersampling.** In notebook 5 undersampling kept 210 good payers out of the 490. Train that
   logistic (all the bad payers plus 210 good ones at random, seed 11), measure its calibration in testing, and apply the
   intercept correction with the constant $\ln(490/210)$. Does it come back to the diagonal as with class weights?
2. **Calibrating on the same rows.** Repeat Platt and isotonic for the random forest with leaves of 1, but
   fitting the calibration function on the **fit** part (the same rows the model was trained
   with) instead of the calibration part. Compare the ECE in testing with the one from section 5. What happened and why?
3. **Calibrating the boosting.** The boosting was already calibrated. Apply `CalibratedClassifierCV` with `cv=5` and
   `method="isotonic"` on the full training set and compare AUC, Brier and ECE in testing with the uncalibrated
   boosting. Did the AUC change? Why, if calibrating does not change the order?



## Solutions


In [ ]:

# Exercise 1: undersampling and intercept correction
rng_b = np.random.default_rng(11)
idx_buenos = rng_b.choice(np.where(y_tr == 0)[0], size=n1, replace=False)
idx_sub = np.r_[np.where(y_tr == 1)[0], idx_buenos]
sub = LogisticRegression(penalty=None, max_iter=5000).fit(X_tr[idx_sub], y_tr[idx_sub])
z_sub = sub.decision_function(X_te)
p_sub = 1 / (1 + np.exp(-z_sub))
p_sub_corr = 1 / (1 + np.exp(-(z_sub - np.log(n0 / n1))))     # ln(490/210): the good payers were underrepresented by that ratio
print(tabla_metricas(y_te, {"undersampling": p_sub, "undersampling + correction": p_sub_corr, "unbalanced": p_base}))
diagrama(y_te, {"undersampling": p_sub, "undersampling + correction": p_sub_corr}, "Exercise 1: undersampling before and after correcting the intercept").show()



The correction works the same way: undersampling is equivalent to weighting the good payers by 210/490, so the
constant is the same $\ln(490/210)$, and the curve comes back close to the diagonal. It stays a bit noisier than
with class weights because the model was trained with 420 rows instead of 700: undersampling throws away information,
and the correction does not recover that.


In [ ]:

# Exercise 2: calibrating on the fit rows (what NOT to do)
rf1 = modelos_tw["random forest, leaves of 1"]
p_fit = rf1.predict_proba(A_fit)[:, 1]
print("ECE of the random forest on its own fit rows:", round(ece(y_fit, p_fit), 3), " | mean p on positives:", round(p_fit[y_fit == 1].mean(), 3), " on negatives:", round(p_fit[y_fit == 0].mean(), 3))
mal_platt = calibrar_platt(p_fit, y_fit, P_te["random forest, leaves of 1"])
mal_iso = calibrar_isotonica(p_fit, y_fit, P_te["random forest, leaves of 1"])
print(tabla_metricas(yt_te, {"uncalibrated": P_te["random forest, leaves of 1"], "Platt on calibration (right)": P_platt["random forest, leaves of 1"], "Platt on fit (wrong)": mal_platt,
                             "isotonic on calibration (right)": P_iso["random forest, leaves of 1"], "isotonic on fit (wrong)": mal_iso}))



On its own fit rows the random forest looks miscalibrated, but the other way around from testing: since
it memorized them, it gives high probabilities to those who defaulted and low probabilities to those who paid, and
reality there is exactly 1 or 0. That is, seen from its own rows the model looks **too
timid** (it says 0.8 and 1 happens; it says 0.1 and 0 happens). A calibration function learned on that learns that
"the model is more reliable than it says" and pushes the testing probabilities toward the extremes, where
they do not belong: the mean probability falls from 0.22 to 0.15, and the ECE and the log loss in testing get worse instead
of better (with isotonic, the log loss explodes). It is the same rule as always in another disguise:
nothing that is learned can be learned on the rows the model already used.


In [ ]:

# Exercise 3: CalibratedClassifierCV on the boosting (full training, 5 folds)
t0 = time.time()
gb_cal = CalibratedClassifierCV(GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=0), method="isotonic", cv=5).fit(A_tr, yt_tr)
p_gb_cal = gb_cal.predict_proba(A_te)[:, 1]
gb_solo = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=0).fit(A_tr, yt_tr)
p_gb = gb_solo.predict_proba(A_te)[:, 1]
print(f"({time.time() - t0:.0f} s)")
print(tabla_metricas(yt_te, {"boosting, full training": p_gb, "CalibratedClassifierCV isotonic, cv=5": p_gb_cal}))



The Brier and the ECE barely move, as befits a model that was already calibrated. But the AUC does
change a little, and that seems to contradict the rule. The explanation is that `CalibratedClassifierCV` with `cv=5`
does not calibrate one model: it trains five boostings on five folds, calibrates each one on the fold that
was left out, and **averages** the five. That average is a small ensemble, and an ensemble can indeed change the
order. When you want to measure only the effect of calibrating, you have to do it by hand as in section 5, with a
single model and a separate calibration split.
